# Lab 2 : Formats de Données et Conversions

Ce notebook démontre les conversions entre différents formats de données :
- **CSV** : Format tabulaire avec séparateurs
- **JSON** : Format d'échange de données structurées
- **XML** : Format hiérarchique avec balises

## 1. Chargement des Données Iris

Chargement du jeu de données Iris et création d'un DataFrame pandas.

In [7]:
# Import des bibliothèques
from sklearn.datasets import load_iris
import pandas as pd

# Chargement du dataset Iris
iris = load_iris()

# Création du DataFrame
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target

# Affichage des premières lignes
print("Colonnes disponibles:", iris.feature_names)
print("\nAperçu des données:")
print(df.head())

Colonnes disponibles: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

Aperçu des données:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  


## 2. Conversion DataFrame → CSV

Conversion du DataFrame au format CSV (Comma-Separated Values).

In [8]:
from io import StringIO

# Conversion en CSV
csv_data = StringIO()
df.to_csv(csv_data, index=False)

# Affichage des premières lignes du CSV
csv_output = csv_data.getvalue()
print("Format CSV:")
print("=" * 50)
print('\n'.join(csv_output.split('\n')[:6]))  # Afficher seulement les 5 premières lignes
print("...")

Format CSV:
sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
5.1,3.5,1.4,0.2,0
4.9,3.0,1.4,0.2,0
4.7,3.2,1.3,0.2,0
4.6,3.1,1.5,0.2,0
5.0,3.6,1.4,0.2,0
...


## 3. Conversion DataFrame → JSON

Conversion du DataFrame au format JSON (JavaScript Object Notation) avec orientation par lignes.

In [9]:
from io import StringIO

# Conversion en JSON (format lines)
json_data = StringIO()
df.to_json(json_data, orient='records', lines=True)

# Affichage des premières lignes du JSON
json_output = json_data.getvalue()
print("Format JSON (lignes):")
print("=" * 50)
print('\n'.join(json_output.split('\n')[:3]))  # Afficher les 3 premières lignes
print("...")

Format JSON (lignes):
{"sepal length (cm)":5.1,"sepal width (cm)":3.5,"petal length (cm)":1.4,"petal width (cm)":0.2,"target":0}
{"sepal length (cm)":4.9,"sepal width (cm)":3.0,"petal length (cm)":1.4,"petal width (cm)":0.2,"target":0}
{"sepal length (cm)":4.7,"sepal width (cm)":3.2,"petal length (cm)":1.3,"petal width (cm)":0.2,"target":0}
...


## 4. Conversion XML → JSON

Lecture d'un fichier XML et conversion en format JSON.
Ce code gère :
- Les **attributs** des éléments XML (convertis en propriétés)
- Le **texte** direct des éléments
- Les **éléments imbriqués** (sous-éléments)

In [10]:
import xml.etree.ElementTree as ET
import json

# Charger le fichier XML
tree = ET.parse("file.xml")
root = tree.getroot()

data = {}

# Parcourir tous les éléments enfants de la racine
for child in root:
    element_data = {}
    
    # Extraire les attributs
    if child.attrib:
        element_data.update(child.attrib)
    
    # Extraire le texte direct
    if child.text and child.text.strip():
        element_data['text'] = child.text.strip()
    
    # Parcourir les sous-éléments
    for subchild in child:
        subchild_data = {}
        
        # Attributs du sous-élément
        if subchild.attrib:
            subchild_data.update(subchild.attrib)
        
        # Texte du sous-élément
        if subchild.text and subchild.text.strip():
            subchild_data['text'] = subchild.text.strip()
        
        # Ajouter au dictionnaire parent
        if subchild_data:
            element_data[subchild.tag] = subchild_data
        elif subchild.text:
            element_data[subchild.tag] = subchild.text
    
    # Gérer les éléments multiples avec le même tag
    if child.tag in data:
        data[child.tag].append(element_data)
    else:
        data[child.tag] = [element_data]

# Convertir en JSON formaté
json_output = json.dumps({root.tag: data}, indent=4, ensure_ascii=False)

print("Conversion XML → JSON:")
print("=" * 50)
print(json_output)

Conversion XML → JSON:
{
    "saranghe": {
        "child": [
            {
                "name": "Frank",
                "test": "0",
                "text": "FRANK likes EVERYONE"
            },
            {
                "name": "Texas",
                "test": "l",
                "text": "TEXAS is a PLACE"
            },
            {
                "name": "Frank",
                "test": "2",
                "text": "Exclusively"
            }
        ],
        "unique": [
            {
                "text": "Add a video URL In here"
            },
            {
                "text": "Add a workbook URL here"
            }
        ],
        "data": [
            {
                "text": "Add the content of your article here",
                "family": {
                    "text": "Add the font family of your text here"
                },
                "size": {
                    "text": "Add the font size of your text here"
                }
            }
    

## 5. Conversion JSON → XML

Lecture d'un fichier JSON et conversion en format XML.
Ce code utilise une approche récursive pour :
- Convertir les propriétés commençant par `@` en **attributs XML**
- Convertir `#text` en **texte d'élément**
- Gérer les **listes** et **objets imbriqués**

In [11]:
import json
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom

# Charger le fichier JSON
with open("data.json", "r", encoding="utf-8") as f:
    json_data = json.load(f)

def dict_to_xml(tag, d):
    """
    Convertir récursivement un dictionnaire en élément XML.
    
    Conventions:
    - Clés commençant par '@' → attributs XML
    - Clé '#text' → texte de l'élément
    - Listes → éléments multiples avec le même tag
    - Dictionnaires → sous-éléments
    """
    elem = ET.Element(tag)
    
    if isinstance(d, dict):
        for key, val in d.items():
            if key.startswith('@'):
                # Attribut XML
                elem.set(key[1:], str(val))
            elif key == '#text':
                # Texte de l'élément
                elem.text = str(val)
            elif isinstance(val, list):
                # Liste d'éléments
                for item in val:
                    child = dict_to_xml(key, item)
                    elem.append(child)
            elif isinstance(val, dict):
                # Sous-élément
                child = dict_to_xml(key, val)
                elem.append(child)
            else:
                # Sous-élément simple avec texte
                child = ET.Element(key)
                child.text = str(val)
                elem.append(child)
    elif isinstance(d, str):
        elem.text = d
    else:
        elem.text = str(d)
    
    return elem

# Créer l'arbre XML à partir du JSON
root_tag = list(json_data.keys())[0]
root = dict_to_xml(root_tag, json_data[root_tag])

# Formater avec indentation
xml_string = ET.tostring(root, encoding='unicode')
dom = minidom.parseString(xml_string)
pretty_xml = dom.toprettyxml(indent="    ")

# Nettoyer les lignes vides
pretty_xml = '\n'.join([line for line in pretty_xml.split('\n') if line.strip()])

print("Conversion JSON → XML:")
print("=" * 50)
print(pretty_xml)

Conversion JSON → XML:
<?xml version="1.0" ?>
<saranghe>
    <child name="Frank" test="0">FRANK likes EVERYONE</child>
    <child name="Texas" test="l">TEXAS is a PLACE</child>
    <child name="Frank" test="2">Exclusively</child>
    <unique>Add a video URL In here</unique>
    <unique>Add a workbook URL here</unique>
    <data>
        Add the content of your article here
        <family>Add the font family of your text here</family>
        <size>Add the font size of your text here</size>
    </data>
</saranghe>
